# 02 — Silver transformations (Databricks)
Runs entirely inside Databricks and consumes the Bronze Delta tables. **Fix:** Binance input can encode epochs in seconds, milliseconds or microseconds; normalize the raw numeric epoch to a real `TIMESTAMP` here, not in Gold. Includes source-file date filters for targeted backfills, data-quality rejection, idempotent MERGE, and the exchangeInfo symbol dimension.

In [ ]:
import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, LongType, StringType, TimestampType,
    DecimalType, IntegerType, ArrayType, BooleanType
)
from delta.tables import DeltaTable

spark.conf.set("spark.sql.session.timeZone", "UTC")
BRONZE_KLINES_TABLE = "workspace.cryptolens.bronze_klines"
BRONZE_EXCHANGE_INFO_TABLE = "workspace.cryptolens.bronze_exchange_info"
SILVER_KLINES_TABLE = "workspace.cryptolens.silver_klines"
SILVER_QUARANTINE_TABLE = "workspace.cryptolens.silver_quarantine"
DIM_SYMBOL_TABLE = "workspace.cryptolens.dim_symbol"
LOG_TABLE = "workspace.cryptolens.pipeline_execution_logs"

LOG_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("parameter_processed", StringType(), False),
    StructField("load_type", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("rows_inserted", LongType(), True),
    StructField("rows_updated", LongType(), True),
    StructField("error_message", StringType(), True),
])

## Parameters
`year/month/day` may be left blank to process all Bronze rows for the selected `load_type` (compatible with older Databricks Jobs), or populated to run an exact historical backfill. Set the same values in Bronze and Silver to avoid reprocessing every previously loaded batch. No local downloading is performed.

In [ ]:
dbutils.widgets.text("interval", "1m")
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental"])
dbutils.widgets.text("year", "")
dbutils.widgets.text("month", "")
dbutils.widgets.text("day", "")
dbutils.widgets.dropdown("require_rows", "true", ["true", "false"])
interval = dbutils.widgets.get("interval").strip()
load_type = dbutils.widgets.get("load_type")
year = dbutils.widgets.get("year").strip()
month = dbutils.widgets.get("month").strip()
day = dbutils.widgets.get("day").strip()
require_rows = dbutils.widgets.get("require_rows") == "true"

if bool(year) != bool(month):
    raise ValueError("year and month must either both be provided or both be blank")
if day and (not year or load_type != "incremental"):
    raise ValueError("day is valid only with year/month and incremental load_type")
if year and (not 1 <= int(month) <= 12 or (day and not 1 <= int(day) <= 31)):
    raise ValueError("Invalid date widget values")

In [ ]:
def utc_now():
    return datetime.now(timezone.utc).replace(tzinfo=None)


def log_execution(layer, parameter, load, start, status, inserted=0, updated=0, error=None):
    row = [(str(uuid.uuid4()), layer, parameter, load, start, utc_now(), status,
            int(inserted), int(updated), str(error)[:1000] if error else None)]
    spark.createDataFrame(row, schema=LOG_SCHEMA).write.format("delta").mode("append").saveAsTable(LOG_TABLE)


def merge_counts(table):
    info = DeltaTable.forName(spark, table).history(1).select("operation", "operationMetrics").first()
    if not info or info["operation"] != "MERGE":
        return 0, 0
    values = info["operationMetrics"] or {}
    return int(values.get("numTargetRowsInserted", 0)), int(values.get("numTargetRowsUpdated", 0))


def bronze_batch():
    if not spark.catalog.tableExists(BRONZE_KLINES_TABLE):
        raise RuntimeError(f"Missing {BRONZE_KLINES_TABLE}; ingest the uploaded CSVs in Bronze first")
    data = spark.table(BRONZE_KLINES_TABLE).filter(
        (F.col("interval") == interval) & (F.col("load_type") == load_type))
    if year:
        suffix = (f"-{interval}-{int(year)}-{int(month):02d}.csv" if load_type == "full" else
                  f"-{interval}-{int(year)}-{int(month):02d}-{int(day):02d}.csv" if day else None)
        if suffix:
            data = data.filter(F.col("source_file").endswith(suffix))
        else:
            # A month-wide incremental backfill: select all days in the month.
            pattern = f"-{interval}-{int(year)}-{int(month):02d}-"
            data = data.filter(F.instr(F.col("source_file"), pattern) > 0)
    return data


def decode_epoch(column):
    # Binance kline archives moved from milliseconds (~1e12) to microseconds
    # (~1e15); older archives may use milliseconds. All conversions preserve
    # microsecond precision and are evaluated on the raw LONG, not a rounded
    # DOUBLE. Spark SQL timestamp_micros/millis are supported on Databricks.
    value = F.col(column)
    magnitude = F.abs(value)
    return (F.when(magnitude >= F.lit(100_000_000_000_000), F.expr(f"timestamp_micros(`{column}`)"))
             .when(magnitude >= F.lit(100_000_000_000), F.expr(f"timestamp_millis(`{column}`)"))
             .otherwise(F.to_timestamp(F.from_unixtime(value))))

## Bronze → Silver OHLCV
Corrected timestamps also change the MERGE keys. Before executing these repaired notebooks for the first time, follow the one-time cleanup in `docs/Databricks_Correction_Run_Guide.md` so historical bad year-58,000 rows are not retained alongside corrected rows.

In [ ]:
def transform_silver_klines(symbol, batch):
    started = utc_now()
    label = f"{symbol}-{interval}-{load_type}" + (f"-{year}-{month}-{day}" if year else "")
    try:
        bronze = batch.filter(F.col("symbol") == symbol)
        converted = (bronze
            .withColumn("open_time", decode_epoch("open_time"))
            .withColumn("close_time", decode_epoch("close_time"))
            .withColumn("open", F.expr("try_cast(`open` AS DECIMAL(18,8))"))
            .withColumn("high", F.expr("try_cast(`high` AS DECIMAL(18,8))"))
            .withColumn("low", F.expr("try_cast(`low` AS DECIMAL(18,8))"))
            .withColumn("close", F.expr("try_cast(`close` AS DECIMAL(18,8))"))
            .withColumn("volume", F.expr("try_cast(`volume` AS DECIMAL(28,8))"))
            .withColumn("quote_asset_volume", F.expr("try_cast(`quote_asset_volume` AS DECIMAL(28,8))"))
            .withColumn("number_of_trades", F.expr("try_cast(`number_of_trades` AS INT)"))
            .withColumn("taker_buy_base_vol", F.expr("try_cast(`taker_buy_base_asset_volume` AS DECIMAL(28,8))"))
            .withColumn("taker_buy_quote_vol", F.expr("try_cast(`taker_buy_quote_asset_volume` AS DECIMAL(28,8))"))
            .withColumn("load_timestamp", F.current_timestamp()))

        # try_cast keeps overflowing or malformed numeric inputs as NULL
        # so the validation below quarantines them under ANSI mode.
        required = ["symbol", "interval", "open_time", "close_time", "open", "high", "low", "close",
                    "volume", "quote_asset_volume", "number_of_trades", "taker_buy_base_vol", "taker_buy_quote_vol"]
        invalid = None
        for field in required:
            cond = F.col(field).isNull()
            invalid = cond if invalid is None else invalid | cond
        invalid = (invalid |
            (F.col("open_time") < F.lit("2010-01-01").cast("timestamp")) |
            (F.col("open_time") >= F.lit("2100-01-01").cast("timestamp")) |
            (F.col("close_time") < F.col("open_time")) |
            (F.col("high") < F.col("low")) |
            (~F.col("open").between(F.col("low"), F.col("high"))) |
            (~F.col("close").between(F.col("low"), F.col("high"))) |
            (F.col("volume") < 0) | (F.col("quote_asset_volume") < 0) |
            (F.col("number_of_trades") < 0))

        bad = converted.filter(invalid)
        rejected = bad.count()
        if rejected:
            (bad.select("symbol", "interval", "open_time", "close_time", "source_file")
                .withColumn("bad_reason", F.lit("SILVER_TYPE_OR_OHLCV_QUALITY"))
                .withColumn("load_timestamp", F.current_timestamp())
                .write.format("delta").mode("append")
                .option("mergeSchema", "true").saveAsTable(SILVER_QUARANTINE_TABLE))

        selected = (converted.filter(~invalid)
            .select("symbol", "interval", "open_time", "close_time", "open", "high", "low", "close",
                    "volume", "quote_asset_volume", "number_of_trades", "taker_buy_base_vol",
                    "taker_buy_quote_vol", "load_timestamp")
            .dropDuplicates(["symbol", "interval", "open_time"]))
        count = selected.count()
        if not count:
            log_execution("Bronze-to-Silver", label, load_type, started, "SUCCESS", 0, 0)
            print(f"[SKIP] {label}: all {rejected} input rows rejected")
            return
        if not spark.catalog.tableExists(SILVER_KLINES_TABLE):
            selected.write.format("delta").saveAsTable(SILVER_KLINES_TABLE)
            inserted, updated = count, 0
        else:
            compare = ["close_time", "open", "high", "low", "close", "volume", "quote_asset_volume",
                       "number_of_trades", "taker_buy_base_vol", "taker_buy_quote_vol"]
            changed = "NOT (" + " AND ".join(f"t.`{name}` <=> s.`{name}`" for name in compare) + ")"
            target = DeltaTable.forName(spark, SILVER_KLINES_TABLE)
            (target.alias("t").merge(selected.alias("s"),
                 "t.symbol = s.symbol AND t.interval = s.interval AND t.open_time = s.open_time")
                 .whenMatchedUpdateAll(condition=changed)
                 .whenNotMatchedInsertAll().execute())
            inserted, updated = merge_counts(SILVER_KLINES_TABLE)
        log_execution("Bronze-to-Silver", label, load_type, started, "SUCCESS", inserted, updated)
        print(f"[OK] {label}: valid={count:,}, rejected={rejected:,}, inserted={inserted:,}, updated={updated:,}")
    except Exception as exc:
        log_execution("Bronze-to-Silver", label, load_type, started, "FAILURE", error=exc)
        raise

In [ ]:
input_batch = bronze_batch()
symbols = [r["symbol"] for r in input_batch.select("symbol").distinct().collect()]
if not symbols:
    msg = f"No Bronze rows matching load_type={load_type}, interval={interval}, date={year}-{month}-{day}"
    log_execution("Bronze-to-Silver", msg, load_type, utc_now(), "FAILURE" if require_rows else "SKIPPED", error=msg)
    if require_rows:
        raise ValueError(msg)
    print("[SKIPPED]", msg)
else:
    print(f"Processing {len(symbols)} symbols from Bronze")
    for symbol in sorted(symbols):
        transform_silver_klines(symbol, input_batch)

## exchangeInfo → Silver dim_symbol
Parse the last *landed* exchangeInfo snapshot using an explicit nested StructType. Update rows only when the actual status/attributes change, and soft-delete symbols absent from a newer snapshot. This retains the Bronze/Silver Medallion boundary for Gold.

In [ ]:
EXCHANGE_INFO_JSON_SCHEMA = StructType([
    StructField("symbols", ArrayType(StructType([
        StructField("symbol", StringType(), True),
        StructField("status", StringType(), True),
        StructField("baseAsset", StringType(), True),
        StructField("quoteAsset", StringType(), True),
        StructField("isSpotTradingAllowed", BooleanType(), True),
    ])), True)
])


def refresh_dim_symbol():
    started = utc_now()
    label = "dim_symbol_refresh"
    try:
        if not spark.catalog.tableExists(BRONZE_EXCHANGE_INFO_TABLE):
            log_execution("Bronze-to-Silver", label, "snapshot", started, "SKIPPED",
                          error="No bronze_exchange_info table")
            print("[SKIPPED] no exchangeInfo snapshot in Bronze")
            return
        snapshots = spark.table(BRONZE_EXCHANGE_INFO_TABLE)
        # Filename dates are ISO-8601, so sorting gives newest source snapshot,
        # regardless of which historical snapshot was uploaded most recently.
        latest = snapshots.orderBy(F.desc("source_file"), F.desc("load_timestamp")).limit(1)
        if not latest.count():
            raise ValueError("bronze_exchange_info contains no snapshots")
        source_file = latest.select("source_file").first()[0]
        staging = (latest
            .select(F.explode(F.from_json(F.col("raw_json"), EXCHANGE_INFO_JSON_SCHEMA).symbols).alias("s"))
            .select(F.col("s.symbol").alias("symbol"), F.col("s.status").alias("status"),
                    F.col("s.baseAsset").alias("base_asset"), F.col("s.quoteAsset").alias("quote_asset"),
                    F.col("s.isSpotTradingAllowed").alias("is_spot_trading_allowed"))
            .filter(F.col("symbol").isNotNull())
            .dropDuplicates(["symbol"])
            .withColumn("load_timestamp", F.current_timestamp()))
        source_count = staging.count()
        if source_count < 1:
            raise ValueError(f"No valid symbols parsed from {source_file}; abort soft-delete")
        if not spark.catalog.tableExists(DIM_SYMBOL_TABLE):
            staging.withColumn("is_active", F.lit(True)).write.format("delta").saveAsTable(DIM_SYMBOL_TABLE)
            inserted, updated = source_count, 0
        else:
            changed = ("NOT (t.status <=> s.status AND t.base_asset <=> s.base_asset AND "
                       "t.quote_asset <=> s.quote_asset AND "
                       "t.is_spot_trading_allowed <=> s.is_spot_trading_allowed AND t.is_active <=> true)")
            (DeltaTable.forName(spark, DIM_SYMBOL_TABLE).alias("t")
                .merge(staging.alias("s"), "t.symbol = s.symbol")
                .whenMatchedUpdate(condition=changed, set={
                    "status": "s.status", "base_asset": "s.base_asset", "quote_asset": "s.quote_asset",
                    "is_spot_trading_allowed": "s.is_spot_trading_allowed", "is_active": "true",
                    "load_timestamp": "s.load_timestamp"})
                .whenNotMatchedInsert(values={
                    "symbol": "s.symbol", "status": "s.status", "base_asset": "s.base_asset",
                    "quote_asset": "s.quote_asset", "is_spot_trading_allowed": "s.is_spot_trading_allowed",
                    "is_active": "true", "load_timestamp": "s.load_timestamp"})
                .whenNotMatchedBySourceUpdate(condition="t.is_active <=> true", set={
                    "status": "'DELISTED'", "is_active": "false", "load_timestamp": "current_timestamp()"})
                .execute())
            inserted, updated = merge_counts(DIM_SYMBOL_TABLE)
        log_execution("Bronze-to-Silver", label, "snapshot", started, "SUCCESS", inserted, updated)
        print(f"[OK] dim_symbol using {source_file}: inserted={inserted}, updated={updated}")
    except Exception as exc:
        log_execution("Bronze-to-Silver", label, "snapshot", started, "FAILURE", error=exc)
        raise

refresh_dim_symbol()